# Smart IDS v3 — Notebook 03: Supervised Multi-Class Models
XGBoost Baseline, Deep Multi-Layer Perceptron (MLP), and 1D-CNN

---

### What You Will Learn
1. **Tree-Based Baseline**: Training an Extreme Gradient Boosting (XGBoost) classifier with histogram split finding.
2. **Deep MLP Architecture**: Building a 4-layer regularized funnel (512-256-128-64) with LeakyReLU, BatchNorm, L2, and Dropout.
3. **1D-CNN Architecture**: Reinterpreting tabular flow features as pseudo-sequences for localized pattern detection.
4. **Adaptive Training Callbacks**: Using `EarlyStopping` (patience 8), `ReduceLROnPlateau`, and `ModelCheckpoint`.
5. **Convergence Analysis**: Comparing learning curves and loss stabilization across architectures.

---

### Prerequisites
- Completion of Notebooks 01 and 02.
- Basic understanding of neural network layers (Dense, Conv1D, BatchNormalization).


## 1. The Supervised Learning Strategy

In this notebook, we train 3 distinct supervised models:
1. **XGBoost Classifier**: Fast, battle-tested decision-tree ensemble baseline for tabular data.
2. **Deep Multi-Layer Perceptron (MLP)**: Deep feedforward neural network designed to capture non-linear relationships across all features.
3. **1D-Convolutional Neural Network (1D-CNN)**: Slides 1D convolutional kernels across the feature vector to detect localized co-occurrences (e.g. adjacent packet timing and flag statistics).


In [1]:
import sys
from pathlib import Path

cur = Path.cwd()
root = cur if (cur / 'src').exists() else cur.parent
if str(root / 'src') not in sys.path:
    sys.path.insert(0, str(root / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import to_categorical

from smart_ids.config import (
    CLASS_NAMES,
    MODELS_DIR,
    PROCESSED_DATA_DIR,
    SEED,
)
from smart_ids.models import build_mlp, build_cnn, build_xgboost
from smart_ids.training import train_mlp, train_cnn, train_xgboost, get_callbacks
from smart_ids.artifacts import save_xgboost_model

# Set seeds
np.random.seed(SEED)
tf.random.set_seed(SEED)
print(f"TensorFlow {tf.__version__} initialized.")


TensorFlow 2.16.2 initialized.


## 2. Loading Preprocessed Data Splits

We load the scaled and SMOTE-rebalanced training data, plus the naturally imbalanced validation and test sets saved by Notebook 02.


In [2]:
X_train_sm = np.load(PROCESSED_DATA_DIR / 'X_train_sm.npy')
y_train_sm = np.load(PROCESSED_DATA_DIR / 'y_train_sm.npy')
X_val_scaled = np.load(PROCESSED_DATA_DIR / 'X_val_scaled.npy')
y_val = np.load(PROCESSED_DATA_DIR / 'y_val.npy')
X_test_scaled = np.load(PROCESSED_DATA_DIR / 'X_test_scaled.npy')
y_test = np.load(PROCESSED_DATA_DIR / 'y_test.npy')

# One-hot encode targets for categorical cross-entropy neural network loss
num_classes = len(CLASS_NAMES)
y_train_cat = to_categorical(y_train_sm, num_classes=num_classes)
y_val_cat = to_categorical(y_val, num_classes=num_classes)

n_features = X_train_sm.shape[1]
print(f"Input Features: {n_features}")
print(f"Training Samples (Balanced via SMOTE): {X_train_sm.shape[0]:,}")
print(f"Validation Samples (Naturally Imbalanced): {X_val_scaled.shape[0]:,}")
print(f"Test Samples (Naturally Imbalanced):       {X_test_scaled.shape[0]:,}")


Input Features: 77
Training Samples (Balanced via SMOTE): 120,208
Validation Samples (Naturally Imbalanced): 3,188
Test Samples (Naturally Imbalanced):       3,750


## 3. Model 1: Extreme Gradient Boosting (XGBoost) Baseline

**What we do and why:**
We train an `XGBClassifier` with 300 estimators, tree depth 6, and histogram split finding (`tree_method='hist'`). We supply the validation set to monitor early stopping with a patience of 20 rounds.


In [3]:
xgb_clf = build_xgboost(num_classes=num_classes, random_state=SEED)
xgb_trained = train_xgboost(xgb_clf, X_train_sm, y_train_sm, X_val_scaled, y_val)

# Save trained XGBoost model to JSON
save_xgboost_model(xgb_trained)
print("XGBoost Baseline trained and saved to models/xgb_model.json.")


XGBoost Baseline trained and saved to models/xgb_model.json.


## 4. Model 2: Deep Multi-Layer Perceptron (MLP)

**What we do and why:**
We build a 4-layer funnel architecture: `512 -> 256 -> 128 -> 64`.
Each block applies:
1. `Dense(units, L2=1e-4, use_bias=False)`
2. `BatchNormalization()` (stabilizes gradient flow)
3. `LeakyReLU(alpha=0.1)` (prevents dying ReLU neurons)
4. `Dropout(0.4)` (prevents co-adaptation and overfitting)
5. `Adam` optimizer with `clipnorm=1.0` (gradient clipping).


In [4]:
mlp_model = build_mlp(input_dim=n_features, num_classes=num_classes)
mlp_callbacks = get_callbacks("mlp", patience_es=8, checkpoint_name="best_mlp.keras")

mlp_trained, mlp_history = train_mlp(
    mlp_model,
    X_train_sm,
    y_train_cat,
    X_val_scaled,
    y_val_cat,
    epochs=15,
    batch_size=1024,
    callbacks=mlp_callbacks,
)
print("MLP Training complete. Best weights saved to models/best_mlp.keras.")


Epoch 1/15


  1/118 ━━━━━━━━━━━━━━━━━━━━ 6:30 3s/step - accuracy: 0.1270 - loss: 2.8010

  5/118 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.1855 - loss: 2.4465

  9/118 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.2497 - loss: 2.2184

 13/118 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.3027 - loss: 2.0446

 17/118 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.3432 - loss: 1.9171

... (463 outputs left out)

116/118 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9831 - loss: 0.0980

118/118 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9831 - loss: 0.0981 - val_accuracy: 0.8974 - val_loss: 0.4538 - learning_rate: 0.0010


Restoring model weights from the end of the best epoch: 14.


MLP Training complete. Best weights saved to models/best_mlp.keras.


## 5. Model 3: 1D-Convolutional Neural Network (1D-CNN)

**What we do and why:**
We reshape the input features to shape `(n_samples, n_features, 1)`. 
Conv1D kernels slide across the feature sequence to capture statistical co-occurrences between adjacent engineered features (like forward/backward packet sizes and inter-arrival times).


In [5]:
cnn_model = build_cnn(input_dim=n_features, num_classes=num_classes)
cnn_callbacks = get_callbacks("cnn", patience_es=4, checkpoint_name="best_cnn.keras")

cnn_trained, cnn_history = train_cnn(
    cnn_model,
    X_train_sm,
    y_train_cat,
    X_val_scaled,
    y_val_cat,
    epochs=12,
    batch_size=2048,
    callbacks=cnn_callbacks,
)
print("1D-CNN Training complete. Best weights saved to models/best_cnn.keras.")


Epoch 1/12


 1/59 ━━━━━━━━━━━━━━━━━━━━ 4:02 4s/step - accuracy: 0.1187 - loss: 2.5810

 2/59 ━━━━━━━━━━━━━━━━━━━━ 28s 502ms/step - accuracy: 0.1301 - loss: 2.4821

 3/59 ━━━━━━━━━━━━━━━━━━━━ 29s 523ms/step - accuracy: 0.1423 - loss: 2.4329

 4/59 ━━━━━━━━━━━━━━━━━━━━ 26s 483ms/step - accuracy: 0.1569 - loss: 2.3851

 5/59 ━━━━━━━━━━━━━━━━━━━━ 25s 468ms/step - accuracy: 0.1685 - loss: 2.3474

... (299 outputs left out)

59/59 ━━━━━━━━━━━━━━━━━━━━ 21s 350ms/step - accuracy: 0.7696 - loss: 0.6117 - val_accuracy: 0.0775 - val_loss: 4.9375 - learning_rate: 0.0010


Epoch 5: early stopping


Restoring model weights from the end of the best epoch: 1.


1D-CNN Training complete. Best weights saved to models/best_cnn.keras.


## 6. Comparing Neural Training Convergence Histories

Let's visualize the loss and accuracy convergence profiles of the MLP and 1D-CNN across epochs.


In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# MLP Loss
axes[0, 0].plot(mlp_history.history['loss'], label='Train Loss', color='#2563eb')
axes[0, 0].plot(mlp_history.history['val_loss'], label='Val Loss', linestyle='--', color='#f59e0b')
axes[0, 0].set_title('MLP — Categorical Cross-Entropy Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# MLP Accuracy
axes[0, 1].plot(mlp_history.history['accuracy'], label='Train Acc', color='#2563eb')
axes[0, 1].plot(mlp_history.history['val_accuracy'], label='Val Acc', linestyle='--', color='#10b981')
axes[0, 1].set_title('MLP — Classification Accuracy')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# CNN Loss
axes[1, 0].plot(cnn_history.history['loss'], label='Train Loss', color='#7c3aed')
axes[1, 0].plot(cnn_history.history['val_loss'], label='Val Loss', linestyle='--', color='#f59e0b')
axes[1, 0].set_title('1D-CNN — Categorical Cross-Entropy Loss')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# CNN Accuracy
axes[1, 1].plot(cnn_history.history['accuracy'], label='Train Acc', color='#7c3aed')
axes[1, 1].plot(cnn_history.history['val_accuracy'], label='Val Acc', linestyle='--', color='#10b981')
axes[1, 1].set_title('1D-CNN — Classification Accuracy')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


<Figure size 1400x900 with 4 Axes>

---
## Summary and Next Notebook

In this notebook:
- We trained an **XGBoost classifier** baseline with early stopping.
- We constructed and trained a regularized **Deep MLP** (512-256-128-64).
- We trained a **1D-CNN** with hierarchical convolutional blocks and global average pooling.
- All three models were persisted to `models/` for downstream benchmarking.

**Next up:** In **Notebook 04: Autoencoder Zero-Day Anomaly Detection**, we will explore why supervised models fail against unseen attacks, and how to train an unsupervised Autoencoder on benign traffic to detect zero-day threats!
